# Análisis exploratorio de datos (EDA)

**Trabajo de grado — Tecnológico de Antioquia**  
*Análisis de modelos de transacción energética y desarrollo de un modelo predictivo para la estimación del precio de la energía*

Este cuaderno explora las series del mercado eléctrico colombiano antes de modelar. El objetivo no es "hacer gráficas bonitas", sino responder preguntas que condicionan las decisiones de modelado:

1. ¿Cómo se distribuye el precio de bolsa? ¿Hay asimetría o valores extremos?
2. ¿Qué estacionalidades existen (horaria, semanal, anual)?
3. ¿Cuánta autocorrelación tiene la serie? → justifica los rezagos
4. ¿Qué variables correlacionan con el precio? → justifica las exógenas
5. ¿Cómo afecta el fenómeno ENSO (El Niño / La Niña)?

> **Antes de ejecutar**: hay que tener datos cargados.  
> `python -m scripts.ingestar_datos --inicio 2024-01-01 --fin 2025-12-31`

In [ ]:
import sys
from pathlib import Path

# Permite importar `src.*` desde el cuaderno
sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (13, 5)
plt.rcParams["figure.dpi"] = 110

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

## 1. Carga de datos

Se leen desde la base de datos (Neon). Si no hay conexión, se usa el CSV de ejemplo.

In [ ]:
try:
    from sqlalchemy import select

    from src.database.models import DatoHistorico
    from src.database.session import SessionLocal

    with SessionLocal() as db:
        filas = db.execute(select(DatoHistorico).order_by(DatoHistorico.fecha_hora)).scalars().all()

    columnas = [c.name for c in DatoHistorico.__table__.columns if c.name != "creado_en"]
    df = pd.DataFrame([{c: getattr(f, c) for c in columnas} for f in filas])
    origen = "base de datos (Neon)"
except Exception as exc:
    print(f"No se pudo leer de la base de datos ({type(exc).__name__}); se usa el CSV de ejemplo")
    df = pd.read_csv(
        Path.cwd().parent / "src" / "data_collection" / "samples" / "xm_historico_ejemplo.csv",
        parse_dates=["fecha_hora"],
    )
    origen = "CSV de ejemplo (SIMULADO)"

df["fecha_hora"] = pd.to_datetime(df["fecha_hora"]).dt.tz_localize(None)
df = df.sort_values("fecha_hora").reset_index(drop=True)

print(f"Origen : {origen}")
print(f"Filas  : {len(df):,}")
print(f"Rango  : {df['fecha_hora'].min()}  ->  {df['fecha_hora'].max()}")
print(f"Fuente : {df['fuente'].unique().tolist()}")
df.head()

In [ ]:
# Calidad de los datos: nulos y estadísticos básicos
resumen = pd.DataFrame(
    {
        "nulos": df.isna().sum(),
        "% nulos": (df.isna().sum() / len(df) * 100).round(2),
        "tipo": df.dtypes.astype(str),
    }
)
display(resumen[resumen["nulos"] > 0] if resumen["nulos"].sum() else "Sin valores nulos")

df.describe().T.round(2)

## 2. La variable objetivo: precio de bolsa

Lo primero es entender su distribución. Si está muy sesgada a la derecha (cola larga de precios altos), conviene saberlo: afecta a la elección de métrica y puede justificar una transformación logarítmica.

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(16, 4.5))

sns.histplot(df["precio_bolsa"], bins=70, kde=True, ax=ejes[0], color="#2b6cb0")
ejes[0].set_title("Distribución del precio de bolsa")
ejes[0].set_xlabel("COP/kWh")

sns.boxplot(y=df["precio_bolsa"], ax=ejes[1], color="#2b6cb0")
ejes[1].set_title("Diagrama de caja")
ejes[1].set_ylabel("COP/kWh")

sns.histplot(np.log1p(df["precio_bolsa"]), bins=70, kde=True, ax=ejes[2], color="#805ad5")
ejes[2].set_title("log(1 + precio) — ¿se normaliza?")

plt.tight_layout()
plt.show()

asimetria = df["precio_bolsa"].skew()
print(f"Asimetría (skewness): {asimetria:.3f}")
print(f"Curtosis           : {df['precio_bolsa'].kurtosis():.3f}")
print(f"Coef. de variación : {df['precio_bolsa'].std() / df['precio_bolsa'].mean():.3f}")
if asimetria > 1:
    print("\n-> Asimetría fuerte a la derecha: cola de precios extremos.")
    print("   Justifica usar RMSE (penaliza errores grandes) además de MAE.")

In [ ]:
# Serie temporal completa + media móvil semanal
fig, eje = plt.subplots(figsize=(15, 5))
eje.plot(df["fecha_hora"], df["precio_bolsa"], lw=0.4, alpha=0.55, label="Precio horario")
eje.plot(
    df["fecha_hora"],
    df["precio_bolsa"].rolling(168, center=True).mean(),
    lw=2.2,
    color="#c53030",
    label="Media móvil 7 días",
)
eje.set_title("Evolución del precio de bolsa nacional")
eje.set_ylabel("COP/kWh")
eje.legend()
plt.tight_layout()
plt.show()

## 3. Estacionalidad

Tres escalas distintas. Cada patrón que aparezca aquí debe tener su variable correspondiente en `src/processing/features.py`.

In [ ]:
df["hora"] = df["fecha_hora"].dt.hour
df["dia_semana"] = df["fecha_hora"].dt.dayofweek
df["mes"] = df["fecha_hora"].dt.month

fig, ejes = plt.subplots(1, 3, figsize=(17, 4.5))

sns.boxplot(data=df, x="hora", y="precio_bolsa", ax=ejes[0], color="#4299e1", fliersize=1)
ejes[0].set_title("Por hora del día")

dias = ["Lun", "Mar", "Mié", "Jue", "Vie", "Sáb", "Dom"]
sns.boxplot(data=df, x="dia_semana", y="precio_bolsa", ax=ejes[1], color="#48bb78", fliersize=1)
ejes[1].set_xticks(range(7))
ejes[1].set_xticklabels(dias)
ejes[1].set_title("Por día de la semana")

sns.boxplot(data=df, x="mes", y="precio_bolsa", ax=ejes[2], color="#ed8936", fliersize=1)
ejes[2].set_title("Por mes (temporada seca: dic–mar)")

for eje in ejes:
    eje.set_ylabel("COP/kWh")
plt.tight_layout()
plt.show()

print("Precio medio por hora del día:")
print(df.groupby("hora")["precio_bolsa"].mean().round(1).to_string())

## 4. Autocorrelación

Esta es la gráfica que **justifica los rezagos**. Si el precio de hace 24 h o de hace 168 h (una semana) sigue correlacionando fuerte con el actual, esos rezagos deben ser variables del modelo.

In [ ]:
serie = df.set_index("fecha_hora")["precio_bolsa"]
rezagos = list(range(1, 200))
autocorr = [serie.autocorr(lag=k) for k in rezagos]

fig, eje = plt.subplots(figsize=(15, 4.5))
eje.bar(rezagos, autocorr, width=0.9, color="#2b6cb0")
for k, etiqueta, color in ((24, "24 h (1 día)", "#c53030"), (168, "168 h (1 semana)", "#38a169")):
    eje.axvline(k, color=color, ls="--", lw=1.6, label=etiqueta)
eje.axhline(0, color="black", lw=0.8)
eje.set_title("Función de autocorrelación del precio de bolsa")
eje.set_xlabel("Rezago (horas)")
eje.set_ylabel("Correlación")
eje.legend()
plt.tight_layout()
plt.show()

print("Autocorrelación en los rezagos usados por el modelo:")
for k in (1, 2, 3, 6, 12, 24, 48, 168):
    print(f"  lag {k:>4} h : {serie.autocorr(lag=k):.4f}")

## 5. Relación con las variables explicativas

¿Qué variables del mercado y del clima se mueven con el precio?

In [ ]:
numericas = df.select_dtypes(include=[np.number]).drop(
    columns=["id", "hora", "dia_semana", "mes"], errors="ignore"
)
numericas = numericas.loc[:, numericas.nunique() > 1]

correlacion = numericas.corr()

fig, eje = plt.subplots(figsize=(12, 9))
mascara = np.triu(np.ones_like(correlacion, dtype=bool))
sns.heatmap(
    correlacion,
    mask=mascara,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    linewidths=0.4,
    annot_kws={"size": 7},
    ax=eje,
)
eje.set_title("Matriz de correlación de Pearson")
plt.tight_layout()
plt.show()

In [ ]:
# Ranking de correlación con el precio
ranking = (
    correlacion["precio_bolsa"]
    .drop("precio_bolsa")
    .sort_values(key=abs, ascending=False)
)

fig, eje = plt.subplots(figsize=(9, 6))
colores = ["#c53030" if v < 0 else "#2b6cb0" for v in ranking.values]
eje.barh(ranking.index, ranking.values, color=colores)
eje.axvline(0, color="black", lw=0.8)
eje.set_title("Correlación de cada variable con el precio de bolsa")
eje.set_xlabel("Coeficiente de Pearson")
eje.invert_yaxis()
plt.tight_layout()
plt.show()

print(ranking.round(4).to_string())
print("\nRecordatorio: correlación no implica causalidad, y Pearson solo mide")
print("relaciones LINEALES. Random Forest y XGBoost pueden explotar relaciones")
print("no lineales que aquí aparecen como correlación baja.")

## 6. Efecto del fenómeno ENSO (El Niño / La Niña)

Hipótesis del trabajo de grado: **El Niño** reduce los aportes hídricos, obliga a generar con térmicas (más caras) y empuja el precio al alza. **La Niña** hace lo contrario.

Clasificación estándar de la NOAA: `ONI ≥ +0.5` → El Niño · `ONI ≤ −0.5` → La Niña.

In [ ]:
if "fenomeno_climatico" in df.columns and df["fenomeno_climatico"].nunique() > 1:
    orden = ["LA_NINA", "NEUTRO", "EL_NINO"]
    presentes = [f for f in orden if f in df["fenomeno_climatico"].unique()]

    fig, ejes = plt.subplots(1, 2, figsize=(15, 4.8))

    sns.boxplot(
        data=df, x="fenomeno_climatico", y="precio_bolsa",
        order=presentes, ax=ejes[0], palette=["#3182ce", "#a0aec0", "#e53e3e"], fliersize=1,
    )
    ejes[0].set_title("Precio según el fenómeno ENSO")
    ejes[0].set_ylabel("COP/kWh")

    if "indice_oni" in df.columns:
        muestra = df.sample(min(4000, len(df)), random_state=42)
        ejes[1].scatter(muestra["indice_oni"], muestra["precio_bolsa"], s=6, alpha=0.28, color="#2b6cb0")
        ejes[1].axvline(0.5, color="#e53e3e", ls="--", lw=1.4, label="Umbral El Niño")
        ejes[1].axvline(-0.5, color="#3182ce", ls="--", lw=1.4, label="Umbral La Niña")
        ejes[1].set_xlabel("Índice ONI")
        ejes[1].set_ylabel("COP/kWh")
        ejes[1].set_title("Precio frente al índice ONI")
        ejes[1].legend()

    plt.tight_layout()
    plt.show()

    display(
        df.groupby("fenomeno_climatico")["precio_bolsa"]
        .agg(["count", "mean", "std", "min", "max"])
        .round(2)
    )
else:
    print("No hay variabilidad en 'fenomeno_climatico' en este conjunto.")
    print("Puede que NOAA no estuviera accesible durante la ingesta.")

## 7. Hidrología y precio

El mecanismo económico central del mercado colombiano: cuando los embalses bajan, el precio sube.

In [ ]:
candidatas = [c for c in ("aportes_hidricos", "volumen_util_embalse", "volumen_util_energia") if c in df.columns]
candidatas = [c for c in candidatas if df[c].nunique() > 1]

if candidatas:
    fig, ejes = plt.subplots(1, len(candidatas), figsize=(7 * len(candidatas), 4.5), squeeze=False)
    for eje, columna in zip(ejes[0], candidatas):
        muestra = df.sample(min(4000, len(df)), random_state=42)
        eje.scatter(muestra[columna], muestra["precio_bolsa"], s=6, alpha=0.25, color="#2b6cb0")
        r = df[columna].corr(df["precio_bolsa"])
        eje.set_title(f"{columna}  (r = {r:.3f})")
        eje.set_xlabel(columna)
        eje.set_ylabel("COP/kWh")
    plt.tight_layout()
    plt.show()
else:
    print("No hay variables hidrológicas con variabilidad en este conjunto.")

## 8. Verificación de las variables construidas

Se aplica el pipeline real (`limpiar` + `construir_features`) y se comprueba que el conjunto queda listo para entrenar.

In [ ]:
from src.processing.features import columnas_features, construir_features, dividir_train_test
from src.processing.limpieza import limpiar

datos = construir_features(limpiar(df.drop(columns=["hora", "dia_semana", "mes"], errors="ignore")), horizonte=24)
features = columnas_features(datos)
train, test = dividir_train_test(datos, 0.2)

print(f"Filas listas para entrenar : {len(datos):,}")
print(f"Número de variables        : {len(features)}")
print(f"Train                      : {len(train):,}  ({train['fecha_hora'].min()} -> {train['fecha_hora'].max()})")
print(f"Test                       : {len(test):,}  ({test['fecha_hora'].min()} -> {test['fecha_hora'].max()})")
print(f"\nSplit cronológico correcto : {train['fecha_hora'].max() < test['fecha_hora'].min()}")
print(f"Sin nulos                  : {datos[features].isna().sum().sum() == 0}")
print(f"\nVariables: {features}")

## 9. Conclusiones del EDA

Rellena esta sección con lo que observes en **tus** datos. Guía de lo que conviene documentar en el trabajo de grado:

| Hallazgo | Implicación para el modelado |
|---|---|
| Asimetría del precio | Elección de métricas (RMSE vs MAE); posible transformación logarítmica |
| Estacionalidad horaria/semanal | Justifica la codificación cíclica seno/coseno |
| Autocorrelación alta en 24 h y 168 h | Justifica los rezagos `precio_lag_24` y `precio_lag_168` |
| Correlación con hidrología | Justifica incluir aportes y volumen de embalses |
| Efecto ENSO | Justifica el índice ONI como variable exógena |

**Siguiente paso** — entrenar y comparar los modelos:

```bash
python -m scripts.entrenar
```